# Precision-focused detector comparison

This report compares the original Isolation Forest with gradient-boosted trees using causal changes relative to each recording's first 60 readings. These are **development validation results**, selected on these same recordings. They do not establish performance on new machines or confirmed physical failures.

The report reads the saved aggregate results. It does not retrain, access test recordings, or change the active model. The first 60 readings per source recording receive no assessment; startup anomalies count as missed. See `docs/improvement-results.md` for the full protocol and limitations.

In [ ]:
import json

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from forge.config import PROJECT_ROOT

result = json.loads((PROJECT_ROOT / "docs/improvement-results.json").read_text())
baseline = result["baseline_validation"]
selected = result["selected"]["metrics"]
pd.DataFrame({"Original Isolation Forest": baseline, "Relative gradient boosting": selected}).loc[
    [
        "precision",
        "recall",
        "f1",
        "fp",
        "fn",
        "false_alarm_onsets",
        "detected_events",
        "unavailable_rows",
    ]
]

## Confusion matrices

Rows are source annotations; columns are persistent model alerts. Normal initialization readings appear in the suppressed/no-alert cell of the full matrix, so the report also provides FPR over **available normal readings**. The improved model has 118 false positives and misses 1,198 annotated anomaly readings.

In [ ]:
fig = make_subplots(
    rows=1, cols=2, subplot_titles=["Original Isolation Forest", "Relative gradient boosting"]
)
for column, metrics in enumerate([baseline, selected], start=1):
    matrix = [[metrics["tn"], metrics["fp"]], [metrics["fn"], metrics["tp"]]]
    fig.add_trace(
        go.Heatmap(
            z=matrix,
            x=["No alert", "Alert"],
            y=["Normal", "Anomaly"],
            text=matrix,
            texttemplate="%{text}",
            colorscale="Blues",
            showscale=False,
        ),
        row=1,
        col=column,
    )
fig.update_yaxes(autorange="reversed", title_text="Source annotation")
fig.update_layout(height=370, title="Development validation: 9,973 unique observations")
fig.show()

## Candidates under the same recall floors

Each plotted candidate meets pooled point recall ≥60%, event recall ≥60%, and point recall ≥20% in every validation group. Candidates with no feasible threshold are retained in the table. Threshold and persistence selection use validation annotations; estimator fitting uses training data only.

In [ ]:
rows = []
for candidate in result["candidates"]:
    best = candidate["best"]
    rows.append(
        {
            "candidate": candidate["name"],
            "feasible": best is not None,
            "precision": best["metrics"]["precision"] if best else None,
            "recall": best["metrics"]["recall"] if best else None,
        }
    )
comparison = pd.DataFrame(rows)
fig = go.Figure(
    go.Scatter(
        x=comparison.recall,
        y=comparison.precision,
        text=comparison.candidate,
        mode="markers",
        marker={"size": 10},
    )
)
fig.add_hline(y=0.95, line_dash="dash", annotation_text="Precision target")
fig.update_layout(height=420, xaxis_title="Point recall", yaxis_title="Point precision")
fig.update_xaxes(range=[0, 1])
fig.update_yaxes(range=[0, 1])
fig.show()
comparison

## Remaining weaknesses

The selected model achieves 95.1% precision at 65.8% point recall. All ten annotated events have at least one new alert onset, but that does not mean the full anomalous intervals were recovered: the other/10 overlap group has only 21.7% point recall. Median detected-event delay is 41.5 seconds.

There are still 26 false alert onsets, or 15.74 per available normal hour. The 1% FPR and two-onsets-per-hour targets are **not met**. The 519 initialization observations are unavailable after deduplication; none are annotated anomalous in this validation set. An already faulty startup can invalidate the reference on another recording.

In [ ]:
pd.DataFrame.from_dict(result["selected"]["full_validation_metrics"]["per_group"], orient="index")[
    [
        "tp",
        "fp",
        "fn",
        "tn",
        "precision",
        "recall",
        "detected_events",
        "events",
        "false_alarm_onsets",
    ]
]

## Reproduction and next evidence

`python -m forge.ml.development` runs the first comparison; `python -m forge.ml.development --followup` runs the declared follow-up. Neither activates a model or opens the test partition. `python -m forge.ml.activation <run_id>` verifies and activates a selected local artifact for the app, leaving the original baseline pointer intact.

The follow-up was motivated by the initial development comparison, so this is iterative model development, not an untouched benchmark. The next scientific step is a separately designed fresh holdout, preferably from a new operating period or asset, with thresholds frozen before scoring.